In [66]:
import numpy as np
import pandas as pd

INPUT_ID = 0
trans_df = pd.read_csv(f"./datasets/input_{INPUT_ID}.csv")
print("SIZE:", trans_df.size)
trans_df.head(5)

SIZE: 1833326


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/05 07:32,70,10042B660,12966,804D3D1D0,10477.34,US Dollar,10477.34,US Dollar,Cash,0
1,2022/09/01 07:11,265216,81898EF90,265216,81898EF90,36.32,Saudi Riyal,36.32,Saudi Riyal,Reinvestment,0
2,2022/09/09 17:09,215278,805A40930,22086,81BA30AA0,429.35,US Dollar,429.35,US Dollar,Cheque,0
3,2022/09/02 12:07,11,814995990,220015,815E66090,14063.43,US Dollar,14063.43,US Dollar,Cheque,0
4,2022/09/01 01:48,133159,80C2C6560,133159,80C4B08B0,158.91,UK Pound,158.91,UK Pound,Credit Card,0


In [67]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/13 23:47]


In [68]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Series([], Name: Bank, dtype: int64)

In [69]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 60923


In [70]:
# Analyze accounts.
accounts_df = pd.read_csv(f"./datasets/accounts_{INPUT_ID}.csv")
print("SIZE:", accounts_df.shape[0])

SIZE: 101237


In [71]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 33151


In [72]:
ranged_trans_usd_sept_df = trans_usd_sept_1st_df\
    .groupby(["From Bank", "Account"])\
    .filter(lambda x: x.groupby(["To Bank", "Account.1"]).size().size >= 5)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 2667


In [73]:
#1. Amount, source and target accounts for transactions of less than 50 USD.

low_profile_transactions = trans_usd_df[trans_usd_df['Amount Paid'] < 50]
low_profile_transactions = low_profile_transactions[['From Bank', 'Account', 'To Bank','Account.1', 'Amount Paid']]
low_profile_transactions.sort_values(by=["From Bank"], ascending=True)

,From Bank,Account,To Bank,Account.1,Amount Paid
130168,0,80023FDC0,15640,802479D10,45.33
22354,0,8018ADB50,256833,817C341C0,4.24
74790,1,80005F620,11,800151650,24.86
54403,1,80F45EAC0,3597,81712E460,2.67
152475,1,800E335A0,252390,814FDA700,42.47
...,...,...,...,...,...
54387,367585,819EFCA80,58361,819EFD130,40.92
100157,370214,819BB6930,18326,819BB6890,45.79
145113,371204,81A1D0420,65009,81A1CFBA0,18.32
35874,374027,81B1B3640,56914,81B1B0060,3.60


In [74]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
max_amount_bank = max_amount_trans_usd.merge(accounts_df, left_on="From Bank", right_on="Bank ID")
max_amount_bank=max_amount_bank[["From Bank", "Account", "Bank Name","Amount Paid"]].drop_duplicates().sort_values(by="Account", ascending=True)
max_amount_bank

,From Bank,Account,Bank Name,Amount Paid
9299,70,10042B660,Willows Thrift,2.573521e+08
19924,3201,8000FB870,Fieldstone Credit Union,4.088200e+02
3488,12,8001004E0,National Bank of Columbus,2.024653e+07
20063,3214,800111750,First Bank of New Orleans,1.239362e+04
13204,1217,800150A40,Bank of Philadelphia,3.833775e+07
...,...,...,...,...
66609,334915,81C02E780,Sea Credit Union,1.000000e-02
20114,3345,81C057480,Arbor Bancorp,7.523720e+03
67138,369394,81C0B35F0,Regents Bancorp,4.672200e+02
59325,160896,81C1672E0,Hilltop Bancorp,6.780570e+03


In [75]:
#3. Source account, payment format, and amount of transactions in period [2022-09-06, 2022-11-06] with amount lower than AVG/100 of period [2022-09-01, 2022-09-05] for the same type of transaction.

avg_amounts_per_type = trans_usd_sept_1st_df.groupby(["Payment Format"])["Amount Paid"].mean().reset_index()
trans_usd_sept_2nd_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/06') & (trans_usd_df["Timestamp"] <= '2022/09/15')]
trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_df.merge(avg_amounts_per_type, left_on=["Payment Format"], right_on=["Payment Format"]).rename(columns={
    "Amount Paid_x": "Amount Paid",
    "Amount Paid_y": "AVG",
})


lower_trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_with_avg_df[trans_usd_sept_2nd_with_avg_df["Amount Paid"] < trans_usd_sept_2nd_with_avg_df["AVG"] * 0.01]
lower_trans_usd_sept_2nd_with_avg_df=lower_trans_usd_sept_2nd_with_avg_df[["From Bank", "Account", "Payment Format", "Amount Paid"]].sort_values(by=["Account", "Amount Paid"], ascending=True)
lower_trans_usd_sept_2nd_with_avg_df

,From Bank,Account,Payment Format,Amount Paid
1177,70,10042B660,Cash,0.10
27195,70,10042B660,Cash,0.11
11421,70,10042B660,Cash,0.21
6451,70,10042B660,Credit Card,0.22
2237,70,10042B660,Cash,0.34
...,...,...,...,...
18103,110777,81BFE1FA0,ACH,3858.99
25228,144315,81BFECAB0,Credit Card,27.39
11823,159289,81C027760,Credit Card,9.12
6607,127980,81C032990,ACH,2624.85


In [76]:
#4. Accounts that match the scatter-gather pattern and where the source account has transferred to more than 5 distinct accounts.

accounts_df = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
account_pairs_df = accounts_df.merge(trans_usd_sept_1st_df, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"], how="inner").rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]
account_pairs_df = account_pairs_df.drop_duplicates().groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
account_pairs_df = account_pairs_df[(account_pairs_df["size"] >= 5)]


from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts

,Bank,Account


In [77]:
trans_usd_sept_1st_df[(trans_usd_sept_1st_df["Account.1"] == "807C60BC0")]

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering


In [78]:
a=trans_usd_sept_1st_df[(trans_usd_sept_1st_df["Account"] == "10042B660")]
a[a["Account.1"] == "806153310"]

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
33349,2022/09/02 19:07,70,10042B660,2692,806153310,2679.16,US Dollar,2679.16,US Dollar,Credit Card,0


In [79]:
#Conversion dates of period [2022-09-01, 2022-09-05] with base USD
#Bitcoin rates taken from investing.com
#Rest of currencies from api.frankfurter.dev
conversion_rates_records = np.rec.array([
           ('2022/09/01', 1.4644, 5.1805, 1.314 , 0.97999, 6.9   , 1.0002, 0.86272, 3.3535, 79.543, 139.34, 20.189, 60.367, 3.75, 1.,  19793.1),
           ('2022/09/02', 1.4691, 5.2035, 1.3141, 0.98175, 6.9035, 1.0011, 0.86468, 3.3755, 79.719, 140.11, 20.085, 60.427, 3.75, 1., 199999. ),
           ('2022/09/03', 1.4691, 5.2056, 1.3138, 0.98207, 6.9046, 1.0013, 0.86478, 3.3791, 79.75 , 140.17, 20.081, 60.471, 3.75, 1.,  19831.4),
           ('2022/09/04', 1.4695, 5.2082, 1.3139, 0.98219, 6.9047, 1.0013, 0.8649 , 3.3815, 79.754, 140.22, 20.084, 60.461, 3.75, 1.,  19952.7),
           ('2022/09/05', 1.4722, 5.1786, 1.3142, 0.98273, 6.9216, 1.0068, 0.86813, 3.4006, 79.816, 140.49, 20.018, 60.737, 3.75, 1.,  20126.1)],
          dtype=[ ('Date', 'O'), ('Australian Dollar', '<f8'), ('Brazil Real', '<f8'), ('Canadian Dollar', '<f8'), ('Swiss Franc', '<f8'), ('Yuan', '<f8'), ('Euro', '<f8'), ('UK Pound', '<f8'), ('Shekel', '<f8'), ('Rupee', '<f8'), ('Yen', '<f8'), ('Mexican Peso', '<f8'), ('Ruble', '<f8'), ('Saudi Riyal', '<f8'), ('US Dollar', '<f8'), ('Bitcoin', '<f8')])
conversion_rates_df = pd.DataFrame.from_records(conversion_rates_records)
conversion_rates_df = conversion_rates_df.set_index("Date")

In [80]:
#5. Count of transactions of period [2022-09-01, 2022-09-05] with type Wire or ACH, having converted amount for that day less than USD 1.
trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= '2022/09/01') & (trans_df["Timestamp"] <= '2022/09/06')]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[(trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df['Amount'] = trans_sept_1st_wire_or_ach_converted_df.apply(lambda row: row['Amount Paid'] / conversion_rates_df[row['Payment Currency']][row["Timestamp"].split(" ")[0]], axis=1)
trans_sept_1st_wire_or_ach_filtered = trans_sept_1st_wire_or_ach_converted_df[trans_sept_1st_wire_or_ach_converted_df['Amount'] < 1.0]
print("SIZE:", trans_sept_1st_wire_or_ach_filtered.shape[0])

SIZE: 228


In [81]:
from pandas.testing import assert_frame_equal
result_q1 = pd.read_csv(f"./output/q1_output_{INPUT_ID}.csv")
q1_comp1=low_profile_transactions.sort_values(by=["From Bank", "Account"], ascending=True).reset_index(drop=True)
q1_comp2=result_q1.sort_values(by=["From Bank", "Account"], ascending=True).reset_index(drop=True).round(2)

assert_frame_equal(q1_comp1, q1_comp2)

In [82]:
result_q2 = pd.read_csv(f"./output/q2_output_{INPUT_ID}.csv")
q2_comp1=max_amount_bank.sort_values(by=["Amount Paid", "Account"], ascending=True).reset_index(drop=True)
q2_comp2=result_q2.sort_values(by=["Amount Paid", "Account"], ascending=True).reset_index(drop=True).round(2)

assert_frame_equal(q2_comp1[q2_comp2.columns], q2_comp2)

In [83]:
result_q3 = pd.read_csv(f"./output/q3_output_{INPUT_ID}.csv")
q3_comp1=lower_trans_usd_sept_2nd_with_avg_df.sort_values(by=["From Bank", "Account", "Payment Format", "Amount Paid"], ascending=True).reset_index(drop=True)
q3_comp2=result_q3.sort_values(by=["From Bank", "Account", "Payment Format", "Amount Paid"], ascending=True).reset_index(drop=True).round(2)
df = q3_comp1[q3_comp2.columns].merge(q3_comp2, on=q3_comp2.columns.tolist(), how='outer', suffixes=['', '_'], indicator=True)
assert_frame_equal(q3_comp1[q3_comp2.columns], q3_comp2)

In [84]:
result_q4 = pd.read_csv(f'./output/q4_output_{INPUT_ID}.csv')
print(result_q4)
from_account_result_q4 = result_q4[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_result_q4 = result_q4[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
q4_comp2=pd.concat([from_account_result_q4, to_account_result_q4]).drop_duplicates()
q4_comp2=q4_comp2.sort_values(by=["Bank", "Account"], ascending=True).reset_index(drop=True)
q4_comp1 = unique_accounts.sort_values(by=["Bank", "Account"], ascending=True).reset_index(drop=True)
q4_comp1['Account'] = q4_comp1['Account'].astype(str)
q4_comp2['Account'] = q4_comp2['Account'].astype(str)
q4_comp1['Bank'] = q4_comp1['Account'].astype(int)
q4_comp2['Bank'] = q4_comp2['Account'].astype(int)
assert_frame_equal(q4_comp1, q4_comp2)

Empty DataFrame
Columns: [From Bank, From Account, To Bank, To Account]
Index: []


In [85]:
df = pd.read_csv(f'./output/q5_output_{INPUT_ID}.csv', header=None)
valor_unico = df.iloc[0, 0]
print(trans_sept_1st_wire_or_ach_filtered.shape[0]==valor_unico) 

True
